# Download of the MERRA-2 data

Hourly MERRA-2 fields 2015–2025 for the European domain from NASA GES DISC, used in the data chapter (comparison with
ERA5) and read by `residuen.ipynb`.

- Needs a free NASA Earthdata account; `earthaccess` takes the login from `~/.netrc` or asks for it.
- The download links come from the GES DISC subsetter. For each of the two collections below, open the dataset at GES DISC,
  choose "Subset / Get Data" with these settings and save the list of links ("Download Links List") as a `.txt` file next
  to this notebook. Keep the start of the file name the subsetter gives it (`subset_M2T1NXFLX_...` and
  `subset_M2T1NXRAD_...`).

| Collection | Variables | Folder |
|---|---|---|
| M2T1NXFLX (tavg1_2d_flx_Nx, version 5.12.4) | SPEED, TLML | `data/MERRA-2-2015-2025` |
| M2T1NXRAD (tavg1_2d_rad_Nx, version 5.12.4) | SWGDN | `data/MERRA-2-rad-2015-2025` |

Both: 1 January 2015 to 31 December 2025; region west −31, south 18, east 45, north 75; regridding bilinear to the 0.25°
grid of GLDAS-2 ("remapbil GLDAS-2_0.25"); output netCDF. Each collection gives 4,018 daily files.
(`analysis/data_comparison_relative.py solar` additionally compares NIRDF + NIRDR of the flux collection, an optional
check that the thesis does not use; add these two variables to the flux list for it.)

Files that are already in the folder are skipped, so the notebook can be run again after an interruption. Afterwards run
`cd analysis && python organize_raw_data.py`.

In [ ]:
import glob
import os
from urllib.parse import parse_qs, urlparse

import earthaccess

LISTS = {"subset_M2T1NXFLX_*.txt": "./data/MERRA-2-2015-2025",
         "subset_M2T1NXRAD_*.txt": "./data/MERRA-2-rad-2015-2025"}
session = None


# download every file of a link list into a folder; the file name is the LABEL parameter of each link, and links without
# one (the documentation link at the top of the list) are skipped
def download_list(list_file, folder):
    global session
    os.makedirs(folder, exist_ok=True)
    urls = [line.strip() for line in open(list_file) if line.strip().startswith("http")]
    failed, new = [], 0
    for i, url in enumerate(urls, 1):
        name = parse_qs(urlparse(url).query).get("LABEL", [None])[0]
        if name is None:
            continue
        path = os.path.join(folder, name)
        if os.path.exists(path):
            continue
        # log in only when something has to be downloaded
        session = session or earthaccess.login().get_session()
        try:
            with session.get(url, stream=True) as r:
                r.raise_for_status()
                with open(path + ".part", "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        f.write(chunk)
            os.replace(path + ".part", path)
            new += 1
        except Exception as e:
            print(f"failed: {name} ({e})")
            failed.append(name)
        if i % 200 == 0:
            print(f"{os.path.basename(list_file)}: {i} of {len(urls)} links")
    print(f"{os.path.basename(list_file)} -> {folder}: {new} downloaded, {len(failed)} failed")
    return failed

In [ ]:
# both collections, each from the newest link list of its kind
for pattern, folder in LISTS.items():
    lists = sorted(glob.glob(pattern))
    assert lists, f"no link list {pattern} next to this notebook (see the instructions above)"
    download_list(lists[-1], folder)